# Gold Layer: Product Dimension Notebook
This notebook extracts unique product hardware configurations from the **Silver Layer** (`silver_clean_ads`) and populates the **Gold Layer** product dimension table (`dim_products`).

## 1. Setup and Imports
Configure project root path and import dependencies alongside the Silver and Gold layer model namespaces.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is available in system path
project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import select, func, cast, String
from sqlalchemy.dialects.sqlite import insert as sqlite_insert
from sqlalchemy.orm import Session
import polars as pl

# Database configuration and model layer namespaces
from app.config import db_engine
from app.models import silver, gold

## 2. Extract Product Specs from Silver Layer
Query clean ads with a valid `baseline_id` and construct a standardized specification summary string.

In [ ]:
with db_engine.connect() as connection:
    df_silver_products = pl.read_database(
        select(
            silver.SilverCleanAd.baseline_id,
            silver.SilverCleanAd.category,
            silver.SilverCleanAd.brand,
            silver.SilverCleanAd.cpu_brand,
            silver.SilverCleanAd.cpu_model,
            silver.SilverCleanAd.ram_gb,
            silver.SilverCleanAd.storage_gb,
            func.concat_ws(
                ', ', 
                func.nullif(silver.SilverCleanAd.cpu_model, 'Brand Not Informed'),
                cast(func.nullif(silver.SilverCleanAd.ram_gb, 0), String) + 'GB RAM',
                cast(func.nullif(silver.SilverCleanAd.storage_gb, 0), String) + 'GB SSD'
            ).label('specs_summary')
        )
        .where(silver.SilverCleanAd.baseline_id.is_not(None)),
        connection=connection
    )

## 3. Deduplicate Product Dimensions
Deduplicate records by `baseline_id` to ensure one row per unique product configuration.

In [ ]:
df_dim_products = (
    df_silver_products
    .unique(subset=['baseline_id'], keep='first')
)

## 4. Upsert into Gold Product Dimension Table (`dim_products`)
Persist unique product configurations into `gold.DimProduct` using SQLite `on_conflict_do_nothing`.

In [ ]:
if not df_dim_products.is_empty():
    with Session(db_engine) as session:
        stmt = sqlite_insert(gold.DimProduct).values(df_dim_products.to_dicts())
        stmt = stmt.on_conflict_do_nothing(index_elements=['baseline_id'])
        session.execute(stmt)
        session.commit()
        print(f"Successfully upserted {len(df_dim_products)} records into dim_products.")
else:
    print("No new product dimension data found to insert.")